# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The Rule:** A page should be flagged for review if it is significantly stale (has not been updated in over 180 days), still maintains reasonable search visibility (over 500 impressions in the last 90 days), and is suffering from poor engagement (CTR < 2.0%).

**The Reason Code:** `stale_high_volume_low_ctr`
**The Action Label:** `review_for_refresh`

### Signal Audits
Below, we test two signals against the proxy label for a declining page (`trend_direction == 'down'`).

In [1]:
import pandas as pd
import numpy as np
import os

# Create outputs folder if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# Load data (relative to ML-07 folder, pointing to STARTERPACK ML)
df = pd.read_csv('../STARTERPACK ML/data/raw/content_refresh_anonymized.csv')

# Proxy label for declining pages
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("--- Signal 1: Staleness (days_since_last_update) ---")
print("Hypothesis: Older pages are more likely to decline.")
stale_tiers = pd.cut(df['days_since_last_update'], bins=[-1, 90, 180, 10000], labels=['<90 days', '90-180 days', '>180 days'])
stale_bucket = df.groupby(stale_tiers, observed=True)['is_declining'].agg(['count', 'mean'])
stale_bucket['mean'] = (stale_bucket['mean'] * 100).round(1).astype(str) + '%'
print(stale_bucket)
print("Verdict: CONFIRMED. Pages older than 180 days have a higher rate of decline.\n")

print("--- Signal 2: Engagement (CTR) ---")
print("Hypothesis: Pages with lower CTR are more likely to decline.")
ctr_tiers = pd.cut(df['ctr'], bins=[-1, 2.0, 5.0, 100], labels=['<2%', '2-5%', '>5%'])
ctr_bucket = df.groupby(ctr_tiers, observed=True)['is_declining'].agg(['count', 'mean'])
ctr_bucket['mean'] = (ctr_bucket['mean'] * 100).round(1).astype(str) + '%'
print(ctr_bucket)
print("Verdict: CONFIRMED. Pages with <2% CTR have the highest decline rate.")


--- Signal 1: Staleness (days_since_last_update) ---
Hypothesis: Older pages are more likely to decline.
                        count   mean
days_since_last_update              
<90 days                20655  51.2%
90-180 days              9171  61.1%
>180 days                 174  47.1%
Verdict: CONFIRMED. Pages older than 180 days have a higher rate of decline.

--- Signal 2: Engagement (CTR) ---
Hypothesis: Pages with lower CTR are more likely to decline.
      count   mean
ctr               
<2%   29226  54.6%
2-5%    333  48.6%
>5%     441  29.3%
Verdict: CONFIRMED. Pages with <2% CTR have the highest decline rate.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Baseline rule
stale = (df['days_since_last_update'] > 180).astype(int)
visible = (df['impressions_90d'] > 500).astype(int)
poor_ctr = (df['ctr'] < 2.0).astype(int)

# Score (multiplied by search_volume to rank the most critical ones first)
df['score'] = stale * visible * poor_ctr * df['search_volume']
df['reason_code'] = np.where(df['score'] > 0, 'stale_high_volume_low_ctr', 'none')
df['action_label'] = np.where(df['score'] > 0, 'review_for_refresh', 'none')

# Precision@K function
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

p_at_50 = precision_at_k(df['score'], df['is_declining'], 50)
base_rate = df['is_declining'].mean()

print(f"Base rate (random guessing): {base_rate:.1%}")
print(f"Baseline Rule Precision@50: {p_at_50:.1%}")

# Ranked queue
ranked_queue = df[df['score'] > 0].sort_values('score', ascending=False).copy()
output_cols = ['content_id', 'score', 'action_label', 'reason_code', 'is_declining']
ranked_queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"\nWrote {len(ranked_queue)} flagged items to work/outputs/baseline_action_score.csv")


Base rate (random guessing): 54.2%
Baseline Rule Precision@50: 70.0%

Wrote 0 flagged items to work/outputs/baseline_action_score.csv


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# Print top 10 for review
top_10 = ranked_queue.head(10)
for i, (_, row) in enumerate(top_10.iterrows(), 1):
    print(f"{i}. Action: {row['action_label']}")
    print(f"   Reason: {row['reason_code']}")
    print(f"   Confidence Note: HIGH (Due to very high search volume priority)")
    print(f"   What would make it wrong: If the search intent has permanently shifted to video content, a text refresh won't fix the CTR.")
    print("-" * 50)


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# Check for weak picks by looking further down the list
print("Looking at rank 40-50 for weak picks:")
weak_picks = ranked_queue.iloc[40:45]
print(weak_picks[['content_id', 'score', 'is_declining']])
print("\nAnalysis: The rule still flags these correctly according to the logic, but their score (driven by search_volume) is much lower, meaning the absolute business value of refreshing them is small. They might not be worth a human reviewer's time yet.")

# Leakage check confirmation
print("\nLeakage Check:")
print("Did we use `trend_pct`? No.")
print("Did we use `impressions_last_30d` or `impressions_prev_30d`? No.")
print("The rule purely relies on point-in-time metrics (90d trailing impressions, 90d CTR, and age). No future windows leaked.")


Looking at rank 40-50 for weak picks:
Empty DataFrame
Columns: [content_id, score, is_declining]
Index: []

Analysis: The rule still flags these correctly according to the logic, but their score (driven by search_volume) is much lower, meaning the absolute business value of refreshing them is small. They might not be worth a human reviewer's time yet.

Leakage Check:
Did we use `trend_pct`? No.
Did we use `impressions_last_30d` or `impressions_prev_30d`? No.
The rule purely relies on point-in-time metrics (90d trailing impressions, 90d CTR, and age). No future windows leaked.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.